# Datathon 2026 - Reproduce best local multi-pipeline model

Notebook này tái lập pipeline dự báo doanh thu sau khi đã được refactor về cấu trúc chính của repo. Flow chạy như một notebook data science bình thường: audit dữ liệu, EDA ngắn, chạy feature/model pipeline local, sinh artifact giải thích được, validate và export `submission.csv`.

Yêu cầu thư mục:

```text
vinuni_hackathon/
  data/
    processed/sales_forecast_submission/
  notebooks/model_reproduce_best_kaggle_solution.ipynb
  src/models/sales_forecasting/
    pipeline/
    components/
```

Không dùng `Revenue/COGS` từ `sample_submission.csv`; nếu đọc sample thì chỉ dùng cột `Date` để kiểm tra thứ tự submit.

## 0. Setup

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 160)

CWD = Path.cwd().resolve()
ROOT = next((p for p in [CWD, *CWD.parents] if (p / 'data').exists() and (p / 'src').exists()), CWD)
FINAL = ROOT / 'src' / 'models' / 'sales_forecasting'
DATA = ROOT / 'data'
ARTIFACTS = DATA / 'processed' / 'sales_forecast_submission' / 'artifacts'
SUBMISSION_PATH = DATA / 'processed' / 'sales_forecast_submission' / 'submission.csv'
ADV = ARTIFACTS / 'final_candidates'

assert FINAL.exists(), f'Missing sales forecasting package: {FINAL}'
assert DATA.exists(), f'Missing data folder: {DATA}'
assert (FINAL / 'pipeline').exists(), 'Missing local pipeline scripts'
assert (FINAL / 'components').exists(), 'Missing local model component modules'

print('ROOT =', ROOT)
print('PACKAGE =', FINAL)
print('DATA  =', DATA)
print('ARTIFACTS =', ARTIFACTS)

## 1. Load data và audit leakage

In [ ]:
def read_csv(name: str, **kwargs) -> pd.DataFrame:
    return pd.read_csv(DATA / name, low_memory=False, **kwargs)

sales = read_csv('sales.csv', parse_dates=['Date'])
orders = read_csv('orders.csv', parse_dates=['order_date'])
order_items = read_csv('order_items.csv')
products = read_csv('products.csv')
payments = read_csv('payments.csv')
web = read_csv('web_traffic.csv', parse_dates=['date'])

sample_dates = None
sample_path = DATA / 'sample_submission.csv'
if sample_path.exists():
    # Leakage guard: only Date is loaded from sample_submission.
    sample_dates = pd.read_csv(sample_path, usecols=['Date'], parse_dates=['Date'])

print('sales:', sales.shape, sales.Date.min().date(), '->', sales.Date.max().date())
print('orders:', orders.shape)
print('order_items:', order_items.shape)
print('products:', products.shape)
print('payments:', payments.shape)
print('web_traffic:', web.shape)
if sample_dates is not None:
    print('sample dates only:', sample_dates.shape, sample_dates.Date.min().date(), '->', sample_dates.Date.max().date())

assert {'Date', 'Revenue', 'COGS'}.issubset(sales.columns)
assert sales[['Revenue', 'COGS']].notna().all().all()

## 2. EDA ngắn: seasonality, trend và horizon

In [ ]:
eda = sales.copy()
eda['year'] = eda.Date.dt.year
eda['month'] = eda.Date.dt.month
eda['dow'] = eda.Date.dt.dayofweek
eda['doy'] = eda.Date.dt.dayofyear

summary = {
    'n_days': len(eda),
    'train_start': str(eda.Date.min().date()),
    'train_end': str(eda.Date.max().date()),
    'revenue_mean': float(eda.Revenue.mean()),
    'cogs_mean': float(eda.COGS.mean()),
    'revenue_cogs_corr': float(eda[['Revenue','COGS']].corr().iloc[0,1]),
}
print(json.dumps(summary, indent=2, ensure_ascii=False))

display(eda.groupby('year')[['Revenue','COGS']].agg(['mean','median','sum']).round(2))
display(eda.groupby('dow')[['Revenue','COGS']].mean().round(2))

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
axes[0].plot(eda.Date, eda.Revenue, lw=1, color='#1f77b4')
axes[0].set_title('Revenue daily history')
axes[1].plot(eda.Date, eda.COGS, lw=1, color='#d62728')
axes[1].set_title('COGS daily history')
for ax in axes:
    ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 3. Helper chạy pipeline local

In [ ]:
def run_script(rel_path: str) -> None:
    script = FINAL / rel_path
    assert script.exists(), f'Missing script: {script}'
    env = os.environ.copy()
    env['PYTHONPATH'] = str(ROOT) + os.pathsep + env.get('PYTHONPATH', '')
    print(f'\n>>> Running {script.relative_to(FINAL)}')
    started = time.time()
    subprocess.run([sys.executable, str(script)], cwd=str(ROOT), env=env, check=True)
    print(f'<<< Done in {time.time() - started:.1f}s')

def run_repo_script(rel_path: str) -> None:
    script = ROOT / rel_path
    assert script.exists(), f'Missing script: {script}'
    env = os.environ.copy()
    env['PYTHONPATH'] = str(ROOT) + os.pathsep + env.get('PYTHONPATH', '')
    print(f'\n>>> Running {rel_path}')
    started = time.time()
    subprocess.run([sys.executable, str(script)], cwd=str(ROOT), env=env, check=True)
    print(f'<<< Done in {time.time() - started:.1f}s')

ARTIFACTS.mkdir(parents=True, exist_ok=True)
ADV.mkdir(parents=True, exist_ok=True)

## 3.1 Generate deterministic Vietnam calendar CSV


In [ ]:
sys.path.insert(0, str(FINAL))
from src.utils.calendar_vn import build_vn_event_calendar

calendar_path = ROOT / 'docs' / 'sales_forecast_submission' / 'vietnam_calendar_events_deterministic_2012_2024.csv'
calendar_df = build_vn_event_calendar(2012, 2024)
calendar_df.to_csv(calendar_path, index=False)
print('generated:', calendar_path)
display(calendar_df.tail())


## 4. Build base recursive/cv ensemble

In [ ]:
run_script('pipeline/forecast_pipeline.py')

for f in ['cv_metrics.csv', 'cv_weights.json', 'submission_cv_ensemble_regime_recovery.csv']:
    p = ARTIFACTS / f
    print(f, p.exists(), p.stat().st_size if p.exists() else None)

display(pd.read_csv(ARTIFACTS / 'cv_metrics.csv').round(4))

## 5. Build regime, legacy và M5-style blends

In [ ]:
run_script('pipeline/build_v4_regime_candidate.py')
run_script('pipeline/build_legacy_blend_regime.py')

# Một số blend script kỳ vọng tên *_regime_recovery. Component này là shape ensemble recovery upper,
# không dùng sample target; copy tên để giữ pipeline tái lập trong package local.
shape = ARTIFACTS / 'submission_shape_ensemble_recovery_upper.csv'
shape_regime = ARTIFACTS / 'submission_shape_ensemble_recovery_upper_regime_recovery.csv'
if shape.exists() and not shape_regime.exists():
    shutil.copy2(shape, shape_regime)

run_script('pipeline/build_m5_style_blend.py')

blend_files = sorted(p.name for p in ARTIFACTS.glob('m5*.csv'))
print('\n'.join(blend_files))


## 6. Build explainable direct model và final 80/20 blend

In [ ]:
run_script('pipeline/explainable_forecast_factory.py')
run_script('pipeline/build_direct_lgb_candidates.py')
run_repo_script('src/visualization/sales_forecast_feature_importance.py')

candidates = sorted(ADV.glob('submission_m5_lgb_direct_blend_*.csv'))
print('advanced candidates:')
for p in candidates:
    print('-', p.name)

## 7. Export final submission

In [ ]:
FINAL_CANDIDATE = ADV / 'submission_m5_lgb_direct_blend_80_20.csv'
assert FINAL_CANDIDATE.exists(), f'Missing final candidate: {FINAL_CANDIDATE}'

submission_path = SUBMISSION_PATH
shutil.copy2(FINAL_CANDIDATE, submission_path)

sub = pd.read_csv(submission_path, parse_dates=['Date'])
print('submission:', submission_path)
print('shape:', sub.shape)
print('date range:', sub.Date.min().date(), '->', sub.Date.max().date())
print('na:', int(sub.isna().sum().sum()))
print('finite:', bool(np.isfinite(sub[['Revenue','COGS']].to_numpy()).all()))
print('min:', sub[['Revenue','COGS']].min().to_dict())

def validate_submission(df: pd.DataFrame) -> None:
    assert list(df.columns) == ['Date', 'Revenue', 'COGS']
    assert len(df) == 548
    assert str(df.Date.min().date()) == '2023-01-01'
    assert str(df.Date.max().date()) == '2024-07-01'
    assert df[['Revenue','COGS']].notna().all().all()
    assert np.isfinite(df[['Revenue','COGS']].to_numpy()).all()
    assert (df[['Revenue','COGS']] > 0).all().all()

validate_submission(sub)
display(sub.head())
display(sub.tail())

## 8. Explainability artifacts

In [ ]:
files = {
    'direct_factory_cv_metrics': ARTIFACTS / 'direct_factory_cv_metrics.csv',
    'direct_factory_feature_group_importance': ARTIFACTS / 'direct_factory_feature_group_importance.csv',
    'direct_factory_shap_importance': ARTIFACTS / 'direct_factory_shap_importance.csv',
    'explainable_report': ARTIFACTS / 'explainable_forecast_factory_report.md',
    'top_features_png': ARTIFACTS / 'figures' / 'top30_features_gain_overall.png',
}
for name, p in files.items():
    print(name, p.exists(), p)

if files['direct_factory_cv_metrics'].exists():
    display(pd.read_csv(files['direct_factory_cv_metrics']).round(4))
if files['direct_factory_feature_group_importance'].exists():
    display(pd.read_csv(files['direct_factory_feature_group_importance']).round(4))
if files['direct_factory_shap_importance'].exists():
    display(pd.read_csv(files['direct_factory_shap_importance']).head(30))

## 9. Optional Kaggle submit

In [ ]:
# Notebook mặc định không tự submit để tránh tốn lượt ngoài ý muốn.
# Khi muốn submit trong notebook, set environment variable DO_KAGGLE_SUBMIT=1 rồi chạy cell này.
if os.environ.get('DO_KAGGLE_SUBMIT') == '1':
    subprocess.run([
        'uv', 'run', 'kaggle', 'competitions', 'submit',
        '-c', 'datathon-2026-round-1',
        '-f', str(SUBMISSION_PATH),
        '-m', 'sales forecasting pipeline 80/20 M5 direct',
    ], cwd=str(ROOT), check=True)
else:
    print('Skip Kaggle submit. Command:')
    print(f"uv run kaggle competitions submit -c datathon-2026-round-1 -f {SUBMISSION_PATH} -m 'sales forecasting pipeline 80/20 M5 direct'")